In [1]:
import os
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\DataScienceProject\\research'

In [2]:
os.chdir("../")
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\DataScienceProject'

In [ ]:
## Data Ingestion COnfig
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [8]:
from src.datascience.constants import *
from src.datascience.utils.main_utils import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config=self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config= DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config

In [9]:
## Component
import urllib.request as request
from src.datascience import logger
import zipfile

class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config
    
    ## Downloading the Zip file
    def download_zip(self):
        if not os.path.exists(self.config.local_data_file):
            filename, header = request.urlretrieve(
                url= self.config.source_URL,
                filename= self.config.local_data_file
            )
            logger.info(f"{filename} Downloaded! with following info: \n{header}")
        else:
            logger.info(f"File already exists")
    
    ## Extracting the Zip file
    def extract_zip(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_file:
            zip_file.extractall(unzip_path)

In [10]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_zip()
    data_ingestion.extract_zip()
except Exception as e:
    raise e

[ 2026-05-12 12:58:21,187 : INFO : main_utils : YAML file: config\config.yaml Loaded Successfully ]
[ 2026-05-12 12:58:21,190 : INFO : main_utils : YAML file: params.yaml Loaded Successfully ]
[ 2026-05-12 12:58:21,192 : INFO : main_utils : YAML file: schema.yaml Loaded Successfully ]
[ 2026-05-12 12:58:21,194 : INFO : main_utils : Create Directory at: artifacts ]
[ 2026-05-12 12:58:21,196 : INFO : main_utils : Create Directory at: artifacts/data_ingestion ]
[ 2026-05-12 12:58:22,228 : INFO : 4022204659 : artifacts/data_ingestion/data.zip Downloaded! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 6542:2445D:A99BC:1E6